# Entrenar DistilBERT en Google Colab (GPU)

Este notebook entrena el clasificador DistilBERT de 5 clases usando GPU gratuita de Colab.

## Instrucciones:
1. Sube este notebook a Google Colab
2. Ve a **Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU (T4)**
3. Sube los CSVs de `data/splits/` cuando se te pida
4. Ejecuta todas las celdas
5. Descarga la carpeta `distilbert_finetuned.zip` al final

In [ ]:
# Verificar que hay GPU disponible
!nvidia-smi
import torch
print(f"\nCUDA disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Ninguna'}")

In [ ]:
# Instalar dependencias
!pip install -q transformers datasets torch scikit-learn pandas

In [ ]:
# Subir los CSVs de entrenamiento, validacion y test
# Necesitas subir 9 archivos desde data/splits/
import os
from google.colab import files

# Crear estructura de directorios
for domain in ["amazon_reviews", "tweets", "financial_news"]:
    os.makedirs(f"data/splits/{domain}", exist_ok=True)

print("Sube los 9 archivos CSV (train.csv, val.csv, test.csv de cada dominio).")
print("Selecciona TODOS los archivos de una vez.\n")
uploaded = files.upload()

# Mover archivos a la estructura correcta
for filename in uploaded:
    for domain in ["amazon_reviews", "tweets", "financial_news"]:
        # Si el nombre empieza con el dominio o split
        for split in ["train", "val", "test"]:
            if filename == f"{domain}_{split}.csv":
                os.rename(filename, f"data/splits/{domain}/{split}.csv")
                print(f"  {filename} -> data/splits/{domain}/{split}.csv")

print("\nSi los archivos no se movieron automaticamente, ejecuta la siguiente celda.")

In [ ]:
# ALTERNATIVA: Si tus archivos se llaman train.csv, val.csv, test.csv
# y los subiste sin renombrar, usa esta celda para moverlos manualmente.
# Descomenta y ajusta segun necesites:

# import shutil
# # Sube de a 3 archivos por dominio:
# # Primera subida: amazon_reviews
# for split in ["train", "val", "test"]:
#     shutil.move(f"{split}.csv", f"data/splits/amazon_reviews/{split}.csv")
#
# # Segunda subida: tweets  
# for split in ["train", "val", "test"]:
#     shutil.move(f"{split}.csv", f"data/splits/tweets/{split}.csv")
#
# # Tercera subida: financial_news
# for split in ["train", "val", "test"]:
#     shutil.move(f"{split}.csv", f"data/splits/financial_news/{split}.csv")

In [ ]:
# Verificar que los archivos estan en su lugar
import pandas as pd
from pathlib import Path

DOMAINS = ["amazon_reviews", "tweets", "financial_news"]

for domain in DOMAINS:
    for split in ["train", "val", "test"]:
        p = Path(f"data/splits/{domain}/{split}.csv")
        if p.exists():
            df = pd.read_csv(p)
            print(f"OK  {domain}/{split}.csv  ({len(df):,} rows)")
        else:
            print(f"FALTA  {domain}/{split}.csv")

In [ ]:
# Cargar datos de entrenamiento, validacion y test
import pandas as pd
from pathlib import Path

DOMAINS = ["amazon_reviews", "tweets", "financial_news"]

def load_data(split: str):
    frames = []
    for domain in DOMAINS:
        p = Path(f"data/splits/{domain}/{split}.csv")
        if p.exists():
            frames.append(pd.read_csv(p))
    df = pd.concat(frames, ignore_index=True).dropna(subset=["text"])
    return df["text"].tolist(), df["label"].tolist()

train_texts, train_labels = load_data("train")
val_texts, val_labels = load_data("val")
test_texts, test_labels = load_data("test")

print(f"Train: {len(train_texts):,}")
print(f"Val:   {len(val_texts):,}")
print(f"Test:  {len(test_texts):,}")

In [ ]:
# Definir el clasificador DistilBERT (mismo codigo del proyecto)
import json
import numpy as np
import torch
from pathlib import Path
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


class _TextDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class DistilBERTClassifier:
    def __init__(self, model_name="distilbert-base-uncased", num_labels=5, max_length=256):
        self.model_name = model_name
        self.num_labels = num_labels
        self.max_length = max_length
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = None
        self.model = None

    def _init_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name, num_labels=self.num_labels
        )
        self.model.to(self.device)

    def fit(self, texts, labels, val_texts=None, val_labels=None, epochs=3, batch_size=32, lr=2e-5):
        self._init_model()
        self.model.train()

        enc = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )
        dataset = _TextDataset(enc, labels)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        optimizer = AdamW(self.model.parameters(), lr=lr)
        total_steps = len(loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
        )

        for epoch in range(epochs):
            total_loss = 0.0
            for i, batch in enumerate(loader):
                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**batch)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                total_loss += loss.item()

                if (i + 1) % 50 == 0:
                    print(f"  Epoch {epoch+1}/{epochs} - Batch {i+1}/{len(loader)} - Loss: {total_loss/(i+1):.4f}")

            avg_loss = total_loss / len(loader)
            print(f"  Epoch {epoch+1}/{epochs} completada - Loss promedio: {avg_loss:.4f}")

        self.model.eval()
        return self

    def predict(self, texts, batch_size=32):
        if isinstance(texts, str):
            texts = [texts]
        all_preds = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=self.max_length, return_tensors="pt"
            )
            enc = {k: v.to(self.device) for k, v in enc.items()}
            with torch.no_grad():
                logits = self.model(**enc).logits
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
        return np.array(all_preds)

    def save(self, path):
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        self.model.save_pretrained(str(path))
        self.tokenizer.save_pretrained(str(path))
        config = {"num_labels": self.num_labels, "max_length": self.max_length}
        with open(path / "classifier_config.json", "w") as f:
            json.dump(config, f)
        print(f"Modelo guardado en {path}")

In [ ]:
# ENTRENAR DISTILBERT (~15-20 min con GPU T4)
import time

clf = DistilBERTClassifier(max_length=256)

print(f"Dispositivo: {clf.device}")
print(f"Entrenando con {len(train_texts):,} textos...\n")

t0 = time.time()
clf.fit(
    train_texts, train_labels,
    val_texts=val_texts, val_labels=val_labels,
    epochs=3,
    batch_size=32,
    lr=2e-5,
)
elapsed = time.time() - t0
print(f"\nEntrenamiento completado en {elapsed/60:.1f} minutos")

In [ ]:
# Evaluar en test set
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Evaluando en test set...")
preds = clf.predict(test_texts)

acc = accuracy_score(test_labels, preds)
f1 = f1_score(test_labels, preds, average="macro")

label_names = ["Very Negative", "Negative", "Neutral", "Positive", "Very Positive"]

print(f"\nAccuracy: {acc:.4f}")
print(f"F1 Macro: {f1:.4f}")
print(f"\n{classification_report(test_labels, preds, target_names=label_names)}")

In [ ]:
# Guardar el modelo entrenado
clf.save("distilbert_finetuned")

In [ ]:
# Comprimir y descargar
import shutil
from google.colab import files

shutil.make_archive("distilbert_finetuned", "zip", ".", "distilbert_finetuned")
print("Descargando distilbert_finetuned.zip...")
print("Descomprime el ZIP en tu carpeta models/ del proyecto.")
files.download("distilbert_finetuned.zip")